# মানব-ইন-দ্য-লুপ: প্রি-অ্যাকশন গেটস, রিস্ক টিয়ারিং, এবং অডিট লগিং

এই পাঠের README মানব-ইন-দ্য-লুপ পরিচয় করিয়ে দেয় একটি সংক্ষিপ্ত অংশ দিয়ে যা ব্যবহারকারীর কাছে `APPROVE` বা `REJECT` জিজ্ঞাসা করে যখন এজেন্ট ইতিমধ্যে একটি প্রতিক্রিয়া তৈরি করেছে। ঐ প্যাটার্ন একটি ভাল শুরু বিন্দু, কিন্তু প্রোডাকশন HITL বাস্তবায়নে সাধারণত তিনটি অতিরিক্ত অংশ প্রয়োজন:

1. একটি **প্রি-অ্যাকশন গেট** যা **আগে** চলে যখন এজেন্ট একটি ঝুঁকিপূর্ণ ধাপ কার্যকর করার আগে, যাতে খরচ, অপরিবর্তনীয়তা, এবং বিলম্ভতা নিয়ন্ত্রণে থাকে।
2. **রিস্ক টিয়ারিং**, যাতে কম-ঝুঁকিপূর্ণ কাজগুলি স্বয়ংক্রিয়ভাবে কার্যকর হয়, মধ্য-ঝুঁকিপূর্ণ কাজগুলি ব্যাচ-অনুমোদিত হয়, এবং মাত্র উচ্চ-ঝুঁকিপূর্ণ কাজগুলি মানুষের উপর ব্লক করে।
3. একটি **অডিট লগ প্লাস রিভিশন লুপ**, যাতে প্রতিটি গেট সিদ্ধান্ত JSONL আকারে রেকর্ড হয়, এবং একটি প্রত্যাখ্যান এজেন্টকে একটি কাঠামোবদ্ধ কারণে পুনরায় অনুরোধ করে শুধুমাত্র `Revising...` মুদ্রণ করার পরিবর্তে।

এই নোটবুক প্রতিটির তৈরি করে একই প্রাথমিক উপাদানের উপর ভিত্তি করে যা `06-system-message-framework.ipynb` এ আছে। এটি সম্পূর্ণ চালানো হয় `DEMO_MODE = True` (কোনো ইন্টারেক্টিভ ইনপুটের প্রয়োজন নেই) অথবা বাস্তব `input()` প্রম্পট ব্যবহার করে যখন `DEMO_MODE = False`। দ্রষ্টব্য: DEMO_MODE এ তৃতীয় লক্ষ্যটির পুনঃচেষ্টা স্ক্রিপ্ট করা বাধ্যতামূলক, তাই লুপ মেকানিক্স সম্পূর্ণ দৃশ্যমান। বাস্তব পুনঃমূল্যায়ন-চালিত পুনঃবর্গীকরণ এর জন্য `DEMO_MODE = False` এবং একটি অপারেটর প্রয়োজন।

**সীমার বাইরে (অন্য পাঠে পরিচালিত):** প্রমাণীকরণ এবং অ্যাক্সেস নিয়ন্ত্রণ (পাঠ 06 README হুমকি 2), টুল-কল মধ্যস্ততা (পাঠ 14 MAF গভীর বিশ্লেষণ), বহু-এজেন্ট বিতর্ক প্যাটার্ন।


In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

# This notebook uses the Azure OpenAI Responses API via the stable /openai/v1/ endpoint.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API.
endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
if not endpoint:
    raise RuntimeError(
        "AZURE_OPENAI_ENDPOINT environment variable is not set. This notebook needs "
        "an Azure OpenAI resource with a model deployment that supports the Responses "
        "API. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_DEPLOYMENT in "
        "your environment or a local .env file, then run `az login`."
    )

deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# Authenticate with Entra ID (run `az login` first). No api_version is needed.
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)

client = OpenAI(
    base_url=f"{endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)



## প্যাটার্ন ১: প্রাক-কর্ম গেট

README এর HITL স্নিপেট প্রথমে এজেন্টকে কল করে, তারপর ব্যবহারকারীকে আউটপুট অনুমোদনের জন্য অনুরোধ করে। এটি একটি **পোস্ট-অ্যাকশন** ফ্লো। এজেন্ট ইতিমধ্যেই সম্পাদন করেছে, তাই LLM কলের খরচ ইতিমধ্যেই প্রদান করা হয়েছে, এবং যে কোনও সাইড ইফেক্ট (ইমেল পাঠানো, ডাটাবেসের একটি সারি লেখা, মন্তব্য পোস্ট করা) ইতিমধ্যেই ঘটেছে।

একটি **প্রি-অ্যাকশন** ফ্লো এজেন্ট ঝুঁকিপূর্ণ ধাপটি চালানোর আগে গেট সংযুক্ত করে। এজেন্ট পদক্ষেপ প্রস্তাব করে, গেট সিদ্ধান্ত নেয় চালানো হবে কি না, এবং শুধুমাত্র অনুমোদনের পরে সাইড ইফেক্ট ঘটে।

| দিক | পোস্ট-অ্যাকশন অনুমোদন (README স্নিপেট) | প্রি-অ্যাকশন গেট (এই নোটবুক) |
|---|---|---|
| কখন অনুমোদন চলে? | এজেন্ট আউটপুট তৈরি করার পরে | যেকোনো সাইড-ইফেক্টের আগে |
| প্রত্যাখ্যানের ক্ষেত্রে LLM খরচ | ইতিমধ্যেই প্রদান করা হয়েছে | শুধুমাত্র প্রস্তাবের জন্য প্রদান, অ্যাকশনের জন্য নয় |
| অপরিবর্তনীয় সাইড ইফেক্ট | সম্ভব (অ্যাকশন ইতিমধ্যেই হয়েছে) | প্রতিরোধ করা হয়েছে |
| অডিট স্পষ্টতা | অনুমোদন একটি প্রিন্ট স্টেটমেন্ট | অনুমোদন একটি টাইমস্ট্যাম্প, অ্যাকশন, কারণসহ JSON রেকর্ড |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## প্যাটার্ন ২: ঝুঁকি স্তরায়ন

প্রতিটি ক্রিয়াকলাপের জন্য মানুষের অনুমোদন প্রয়োজন হয় না। একটি পাবলিক API এর বিরুদ্ধে কেবলমাত্র-পড়ার জন্য অনুসন্ধান হওয়া একটি গ্রাহক ইমেইল প্রেরণ করার থেকে ভিন্ন ঝুঁকি বহন করে। উভয়কে একইভাবে দেখা অপারেটরের মনোযোগ নষ্ট করে এবং এজেন্টকে ধীর করে।

একটি সহজ ৩-স্তর মডেল:

| স্তর | উদাহরণ | অনুমোদন প্রবাহ |
|---|---|---|
| `low` (কেবল-পড়ার) | একটি নলেজ বেস খোঁজা, ফ্লাইটের অপশন খোঁজা, একটি পাবলিক ওয়েব পেজ আনা | স্বয়ংক্রিয়ভাবে চালানো, নিরীক্ষণের জন্য লগ সংরক্ষণ |
| `medium` (সস্তা পরিবর্তন) | একটি ফলাফল ক্যাশ করা, কাউন্টার বাড়ানো, একটি রিমাইন্ডার নির্ধারণ | স্বয়ংক্রিয়ভাবে চালানো, তবে দৈনন্দিন পর্যালোচনার জন্য একত্রিত করা হয় |
| `high` (বহিঃসম্মুখ বা অপরিবর্তনীয়) | একটি ইমেইল পাঠানো, কার্ড চার্জ করা, একটি পাবলিক চ্যানেলে পোস্ট করা | মানুষের অনুমোদনে বাধা দেওয়া |

এটি একটি স্তরায়ন পদ্ধতি। প্রোডাকশন সিস্টেম সাধারণত আরও সূক্ষ্ম স্তর ব্যবহার করে (যেমন, AWS IAM অনুমতি স্তর, ভূমিকা-ভিত্তিক প্রবেশ স্তর)। নিচের ৩-স্তর সংস্করণটি এমন একটি এজেন্টের জন্য সবচেয়ে ছোট উপকারী সংস্করণ যা শুধুমাত্র-পড়ার এবং পার্শ্বপ্রতিক্রিয়াশীল ক্রিয়াকলাপ মেশায়।

নিচের শ্রেণীবিন্যাসকারী কীওয়ার্ড নিয়মাবলীর উপর ভিত্তি করে যাতে ডেমো নির্ধারিত এবং সস্তা থাকে। একটি প্রোডাকশন সিস্টেমে আপনি এটি একটি শেখা শ্রেণীবিন্যাসকারী বা একটি নীতি ইঞ্জিন দিয়ে প্রতিস্থাপন করবেন।


In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## প্যাটার্ন ৩: অডিট লগ এবং সংশোধন লুপ

একটি `print("Response approved.")` হল অডিট লগ নয়। বিশ্বাসের জন্য, প্রতিটি গেট সিদ্ধান্ত একটি গঠিত ইভেন্ট হিসেবে রেকর্ড করা উচিত যাকে আপনি পরে অনুধাবন, পুনরায় চালনা, বা একটি ঘটনা পর্যালোচনার সঙ্গে সংযুক্ত করতে পারেন।

দুটি অংশ:

১. **সেবলিখিত JSONL।** প্রতিটি সিদ্ধান্তের জন্য একটি লাইন, যার মধ্যে সময়মুদ্রা, ক্রিয়া, স্তর, সিদ্ধান্ত, কারণ থাকে। সহজে গ্রেপ করা যায়, পরে একটি প্রকৃত লগ স্টোরে পাঠানো সহজ।
২. **অস্মতি ক্ষেত্রে সংশোধন লুপ।** যখন গেট `deny` ফেরত দেয়, এজেন্ট নিজেকে অস্বীকারের কারণ নিয়ে পুনরায় প্রম্পট করে, যাতে পরবর্তী প্রস্তাব সমস্যাটি এড়াতে পারে।


In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.responses.create(
        model=deployment,
        input=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_text},
        ],
        store=False,
    )
    return response.output_text.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}



In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## অতিরিক্ত সম্পদসমূহ

একাধিক অন্যান্য পাবলিক প্রকল্প এই HITL প্যাটার্নগুলির বিভিন্ন রূপ বাস্তবায়ন করে। আপনার স্ট্যাকের জন্য কোনটি উপযুক্ত তা খুঁজে পেতে পন্থাগুলোর তুলনা করুন:

- **LangChain** হিউম্যান-ইন-দ্য-লুপ টুল র‍্যাপারস ([docs](https://python.langchain.com/docs/integrations/tools/human_tools)): এমন টুল র‍্যাপারস যা মানব ইনপুটের জন্য এক্সিকিউশন থামায়।
- **AutoGen** `UserProxyAgent` ([v0.2 docs](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ এ এটি পুনর্গঠন করা হয়েছে): বহু-এজেন্ট আলাপচারিতায় মানব প্রতিনিধিত্ব করার জন্য একটি এজেন্ট রোল ব্যবহার করে।
- **Microsoft Agent Framework (MAF)** ফাংশন-ইনভোকেশন মিডলওয়্যার ([docs](https://learn.microsoft.com/agent-framework/)): এমন মিডলওয়্যার যা প্রতি টুল/ফাংশন কলের আশেপাশে চলে, গেটিং লজিক এবং অনুমোদন প্রবাহের জন্য উপযোগী।

প্রতিটি প্রকল্প তিনটি সাব-প্যাটার্ন আলাদাভাবে পরিচালনা করে: LangChain এগুলোকে টুল হিসাবে র‍্যাপ করে, AutoGen একটি এজেন্ট রোল ব্যবহার করে, এবং Microsoft Agent Framework ফাংশন-ইনভোকেশন মিডলওয়্যার ব্যবহার করে। নিজের এজেন্টের জন্য ডিজাইন বাছাই করার আগে এক বা দুটি ইমপ্লিমেন্টেশন পুরোটা পড়ুন।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা শুদ্ধতার জন্য চেষ্টা করি, অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার স্বভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে প্রয়োজনীয় ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
